# HRB Volatility Seasonality Study

## H&R Block — The Most Seasonal Stock in the Market?

**The thesis:** H&R Block's revenue is overwhelmingly concentrated in tax season (Jan–Apr). This creates a structural seasonality in realized volatility that is distinct from broad equity seasonality.

### HRB Fiscal Calendar (FY ends April 30)

| Fiscal Quarter | Calendar Months | Season | Revenue Share |
|---|---|---|---|
| FQ1 | May – Jul | Dead zone | ~5–8% |
| FQ2 | Aug – Oct | Dead zone | ~5–8% |
| FQ3 | Nov – Jan | Tax ramp | ~15–20% |
| FQ4 | Feb – Apr | Peak tax | ~65–75% |

This notebook asks:

1. **Does HRB realized vol exhibit calendar-month seasonality** beyond what SPY/IWM show?
2. **Is vol compressed in the dead zone** (May–Oct) and elevated during tax season?
3. **Do earnings reactions differ by fiscal quarter?** (A miss in FQ4 matters more than FQ1)
4. **Is HRB vol ratio (HRB / IWM) itself seasonal?** If so, this is tradeable.

### Controls

- **SPY** — broad market seasonality baseline
- **IWM** — small-cap peer group (HRB is ~$8B market cap)

---


## 1. Configuration


In [ ]:
# ======================================================================
#  CONFIGURATION
# ======================================================================

TICKERS = ['HRB', 'SPY', 'IWM']

START_DATE = '2010-01-01'
END_DATE   = None

# Rolling RV windows
RV_WINDOW  = 20   # 1-month realized vol
RV_WINDOW2 = 60   # 3-month realized vol
ANN_FACTOR = 252

# HRB fiscal year ends April 30
# Map calendar month -> HRB fiscal quarter
MONTH_TO_FQ = {
    5: 'FQ1', 6: 'FQ1', 7: 'FQ1',
    8: 'FQ2', 9: 'FQ2', 10: 'FQ2',
    11: 'FQ3', 12: 'FQ3', 1: 'FQ3',
    2: 'FQ4', 3: 'FQ4', 4: 'FQ4',
}

FQ_LABELS = {
    'FQ1': 'FQ1 (May–Jul)\nDead Zone',
    'FQ2': 'FQ2 (Aug–Oct)\nDead Zone',
    'FQ3': 'FQ3 (Nov–Jan)\nTax Ramp',
    'FQ4': 'FQ4 (Feb–Apr)\nPeak Tax',
}

MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print(f"Tickers: {TICKERS}")
print(f"Date range: {START_DATE} to {END_DATE or 'latest'}")
print(f"RV windows: {RV_WINDOW}d, {RV_WINDOW2}d")


## 2. Setup


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

COLORS = {
    'hrb': '#1B5E20',
    'spy': '#00C4E7',
    'iwm': '#F6852B',
    'accent': '#182B40',
    'neutral': '#5A5A5A',
    'green': '#6DF2A1',
    'gold': '#F7C940',
    'pink': '#EC3586',
    'tax_season': '#4CAF50',
    'dead_zone': '#BDBDBD',
}

FQ_COLORS = {
    'FQ1': '#BDBDBD',
    'FQ2': '#9E9E9E',
    'FQ3': '#66BB6A',
    'FQ4': '#1B5E20',
}


## 3. Fetch Price Data


In [ ]:
print(f"Fetching {TICKERS} from Yahoo Finance...")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=True)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close']
else:
    prices = raw[['Close']]
    prices.columns = TICKERS

prices = prices.dropna(how='all')
print(f"\nFetched {prices.shape[0]} trading days")
print(f"  {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nMissing values per ticker:")
print(prices.isnull().sum().to_string())


## 4. Compute Returns & Realized Volatility


In [ ]:
# Log returns
log_ret = np.log(prices / prices.shift(1))

# Rolling realized vol (annualized %)
rv20 = log_ret.rolling(RV_WINDOW).apply(
    lambda x: np.sqrt(np.sum(x**2) / len(x)) * np.sqrt(ANN_FACTOR) * 100
)
rv20.columns = [f'{t}_rv20' for t in rv20.columns]

rv60 = log_ret.rolling(RV_WINDOW2).apply(
    lambda x: np.sqrt(np.sum(x**2) / len(x)) * np.sqrt(ANN_FACTOR) * 100
)
rv60.columns = [f'{t}_rv60' for t in rv60.columns]

# Combine into one dataframe
df = pd.concat([log_ret, rv20, rv60], axis=1)
df.columns = (
    [f'{t}_ret' for t in TICKERS] +
    [f'{t}_rv20' for t in TICKERS] +
    [f'{t}_rv60' for t in TICKERS]
)

# Calendar features
df['month'] = df.index.month
df['year'] = df.index.year
df['month_name'] = df.index.strftime('%b')
df['fq'] = df['month'].map(MONTH_TO_FQ)

# Vol ratios
df['HRB_vs_IWM_rv20'] = df['HRB_rv20'] / df['IWM_rv20']
df['HRB_vs_SPY_rv20'] = df['HRB_rv20'] / df['SPY_rv20']
df['HRB_vs_IWM_rv60'] = df['HRB_rv60'] / df['IWM_rv60']

print(f"Panel: {len(df)} rows")
print(f"\nRV20 ranges (ann %):")
for t in TICKERS:
    col = f'{t}_rv20'
    print(f"  {t}: [{df[col].min():.1f}, {df[col].max():.1f}]  median = {df[col].median():.1f}")


## 5. HRB Time Series Overview


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# -- Price
ax = axes[0]
ax.plot(prices.index, prices['HRB'], color=COLORS['hrb'], lw=1)
ax.set_ylabel('Price ($)')
ax.set_title('HRB: Price')

# Shade tax seasons (Feb-Apr each year)
for yr in range(prices.index[0].year, prices.index[-1].year + 1):
    ax.axvspan(pd.Timestamp(yr, 2, 1), pd.Timestamp(yr, 4, 30),
               alpha=0.08, color=COLORS['tax_season'], zorder=0)

# -- RV20 comparison
ax = axes[1]
ax.plot(df.index, df['HRB_rv20'], color=COLORS['hrb'], lw=0.8, alpha=0.9, label='HRB')
ax.plot(df.index, df['IWM_rv20'], color=COLORS['iwm'], lw=0.8, alpha=0.6, label='IWM')
ax.plot(df.index, df['SPY_rv20'], color=COLORS['spy'], lw=0.8, alpha=0.6, label='SPY')
ax.set_ylabel('RV20 (ann %)')
ax.set_title('20-Day Realized Volatility: HRB vs Controls')
ax.legend()

for yr in range(prices.index[0].year, prices.index[-1].year + 1):
    ax.axvspan(pd.Timestamp(yr, 2, 1), pd.Timestamp(yr, 4, 30),
               alpha=0.08, color=COLORS['tax_season'], zorder=0)

# -- Vol ratio
ax = axes[2]
ax.plot(df.index, df['HRB_vs_IWM_rv20'], color=COLORS['accent'], lw=0.8, alpha=0.8)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
ax.axhline(df['HRB_vs_IWM_rv20'].median(), color=COLORS['iwm'], ls='-', lw=1, alpha=0.5,
           label=f'Median = {df["HRB_vs_IWM_rv20"].median():.2f}')
ax.set_ylabel('HRB RV20 / IWM RV20')
ax.set_title('HRB Vol Ratio vs IWM')
ax.legend()

for yr in range(prices.index[0].year, prices.index[-1].year + 1):
    ax.axvspan(pd.Timestamp(yr, 2, 1), pd.Timestamp(yr, 4, 30),
               alpha=0.08, color=COLORS['tax_season'], zorder=0)

plt.tight_layout()
plt.show()
print("Green shading = tax season (Feb–Apr)")


## 6. Monthly Seasonality — Realized Vol by Calendar Month

The core question: does HRB's realized vol have a month-of-year pattern that differs from broad equity?


In [ ]:
monthly_rv = df.dropna(subset=['HRB_rv20']).groupby('month').agg(
    HRB_rv20_mean=('HRB_rv20', 'mean'),
    HRB_rv20_med=('HRB_rv20', 'median'),
    IWM_rv20_mean=('IWM_rv20', 'mean'),
    IWM_rv20_med=('IWM_rv20', 'median'),
    SPY_rv20_mean=('SPY_rv20', 'mean'),
    SPY_rv20_med=('SPY_rv20', 'median'),
    n=('HRB_rv20', 'count'),
).round(2)

print("Monthly Median RV20 (ann %):\n")
print(monthly_rv.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

months = range(1, 13)
x = np.arange(12)
w = 0.25

# -- Panel 1: Median RV20 by month
ax = axes[0]
hrb_vals = [monthly_rv.loc[m, 'HRB_rv20_med'] for m in months]
iwm_vals = [monthly_rv.loc[m, 'IWM_rv20_med'] for m in months]
spy_vals = [monthly_rv.loc[m, 'SPY_rv20_med'] for m in months]

ax.bar(x - w, hrb_vals, w, label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, iwm_vals, w, label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, spy_vals, w, label='SPY', color=COLORS['spy'], alpha=0.7)

# Shade tax season months
for m_idx in [0, 1, 2, 3]:  # Jan-Apr
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Median RV20 (ann %)')
ax.set_title('Median 20-Day Realized Vol by Calendar Month')
ax.legend()

# -- Panel 2: HRB excess vol over IWM by month
ax = axes[1]
excess = [hrb_vals[i] - iwm_vals[i] for i in range(12)]
bar_colors = [COLORS['hrb'] if e > 0 else COLORS['dead_zone'] for e in excess]
ax.bar(x, excess, 0.6, color=bar_colors, alpha=0.8)
ax.axhline(0, color='black', lw=1)

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB Median RV20 − IWM Median RV20 (pp)')
ax.set_title('HRB Excess Volatility Over IWM by Month')

plt.tight_layout()
plt.show()
print("Green shading = tax season months (Jan–Apr)")


## 7. Monthly Seasonality — Returns

Does HRB show return seasonality in addition to vol seasonality? Tax season earnings drive the stock — do returns cluster?


In [ ]:
# Monthly simple returns (sum of daily log returns per calendar month)
monthly_ret = df.groupby([df.index.year, df.index.month]).agg(
    HRB_ret=('HRB_ret', 'sum'),
    IWM_ret=('IWM_ret', 'sum'),
    SPY_ret=('SPY_ret', 'sum'),
)
monthly_ret.index.names = ['year', 'month']
monthly_ret = monthly_ret.reset_index()
# Convert log returns to simple returns for display
for col in ['HRB_ret', 'IWM_ret', 'SPY_ret']:
    monthly_ret[col] = (np.exp(monthly_ret[col]) - 1) * 100  # percent

monthly_ret_season = monthly_ret.groupby('month').agg(
    HRB_mean=('HRB_ret', 'mean'),
    HRB_med=('HRB_ret', 'median'),
    HRB_std=('HRB_ret', 'std'),
    HRB_pct_pos=('HRB_ret', lambda x: (x > 0).mean() * 100),
    IWM_mean=('IWM_ret', 'mean'),
    IWM_med=('IWM_ret', 'median'),
    SPY_mean=('SPY_ret', 'mean'),
    SPY_med=('SPY_ret', 'median'),
    n=('HRB_ret', 'count'),
).round(2)

print("Monthly Return Seasonality (%):\n")
print(monthly_ret_season.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

x = np.arange(12)
w = 0.25

# -- Panel 1: Mean monthly returns
ax = axes[0]
hrb_r = [monthly_ret_season.loc[m, 'HRB_mean'] for m in range(1, 13)]
iwm_r = [monthly_ret_season.loc[m, 'IWM_mean'] for m in range(1, 13)]
spy_r = [monthly_ret_season.loc[m, 'SPY_mean'] for m in range(1, 13)]

ax.bar(x - w, hrb_r, w, label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, iwm_r, w, label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, spy_r, w, label='SPY', color=COLORS['spy'], alpha=0.7)
ax.axhline(0, color='black', lw=1)

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Mean Monthly Return (%)')
ax.set_title('Mean Monthly Return by Calendar Month')
ax.legend()

# -- Panel 2: HRB % positive months
ax = axes[1]
pct_pos = [monthly_ret_season.loc[m, 'HRB_pct_pos'] for m in range(1, 13)]
bar_colors = [COLORS['hrb'] if p > 50 else COLORS['pink'] for p in pct_pos]
ax.bar(x, pct_pos, 0.6, color=bar_colors, alpha=0.8)
ax.axhline(50, color='black', ls='--', lw=1, alpha=0.5, label='50%')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('% of Months Positive')
ax.set_title('HRB: Win Rate by Calendar Month')
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.show()


## 8. Fiscal Quarter Analysis

Aggregating by HRB fiscal quarter to see the dead-zone vs tax-season contrast.


In [ ]:
fq_stats = df.dropna(subset=['HRB_rv20']).groupby('fq').agg(
    n=('HRB_rv20', 'count'),
    HRB_rv20_med=('HRB_rv20', 'median'),
    HRB_rv20_mean=('HRB_rv20', 'mean'),
    HRB_rv20_p25=('HRB_rv20', lambda x: np.percentile(x, 25)),
    HRB_rv20_p75=('HRB_rv20', lambda x: np.percentile(x, 75)),
    IWM_rv20_med=('IWM_rv20', 'median'),
    SPY_rv20_med=('SPY_rv20', 'median'),
    HRB_ret_mean=('HRB_ret', lambda x: x.mean() * ANN_FACTOR * 100),
    vol_ratio_med=('HRB_vs_IWM_rv20', 'median'),
).reindex(['FQ1', 'FQ2', 'FQ3', 'FQ4']).round(2)

print("HRB Fiscal Quarter Summary:\n")
print(fq_stats.to_string())


In [ ]:
fq_order = ['FQ1', 'FQ2', 'FQ3', 'FQ4']

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# -- Panel 1: RV20 distributions by FQ (boxplot style)
ax = axes[0]
fq_data = [df.dropna(subset=['HRB_rv20']).loc[df.fq == fq, 'HRB_rv20'].values for fq in fq_order]
bp = ax.boxplot(fq_data, positions=range(4), widths=0.5, patch_artist=True,
                showfliers=False, medianprops={'color': 'black', 'lw': 2})
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(FQ_COLORS[fq_order[i]])
    patch.set_alpha(0.7)
ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('HRB RV20 (ann %)')
ax.set_title('HRB RV20 Distribution by Fiscal Quarter')

# -- Panel 2: HRB vs IWM vol ratio by FQ
ax = axes[1]
ratio_data = [df.dropna(subset=['HRB_vs_IWM_rv20']).loc[df.fq == fq, 'HRB_vs_IWM_rv20'].values for fq in fq_order]
bp2 = ax.boxplot(ratio_data, positions=range(4), widths=0.5, patch_artist=True,
                 showfliers=False, medianprops={'color': 'black', 'lw': 2})
for i, patch in enumerate(bp2['boxes']):
    patch.set_facecolor(FQ_COLORS[fq_order[i]])
    patch.set_alpha(0.7)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('HRB RV20 / IWM RV20')
ax.set_title('HRB Relative Vol (vs IWM) by Fiscal Quarter')

# -- Panel 3: Median comparison HRB vs controls
ax = axes[2]
x = np.arange(4)
w = 0.25
ax.bar(x - w, [fq_stats.loc[fq, 'HRB_rv20_med'] for fq in fq_order], w,
       label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, [fq_stats.loc[fq, 'IWM_rv20_med'] for fq in fq_order], w,
       label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, [fq_stats.loc[fq, 'SPY_rv20_med'] for fq in fq_order], w,
       label='SPY', color=COLORS['spy'], alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('Median RV20 (ann %)')
ax.set_title('Median RV20 by Fiscal Quarter: HRB vs Controls')
ax.legend()

plt.tight_layout()
plt.show()


## 9. Vol Ratio Seasonality — Is HRB/IWM Itself Seasonal?

If the vol ratio (HRB RV20 / IWM RV20) is systematically higher in certain months, that's a tradeable signal: sell HRB vol relative to index vol in the dead zone, buy it into tax season.


In [ ]:
vol_ratio_monthly = df.dropna(subset=['HRB_vs_IWM_rv20']).groupby('month').agg(
    ratio_mean=('HRB_vs_IWM_rv20', 'mean'),
    ratio_med=('HRB_vs_IWM_rv20', 'median'),
    ratio_p25=('HRB_vs_IWM_rv20', lambda x: np.percentile(x, 25)),
    ratio_p75=('HRB_vs_IWM_rv20', lambda x: np.percentile(x, 75)),
    n=('HRB_vs_IWM_rv20', 'count'),
).round(3)

print("HRB/IWM Vol Ratio by Calendar Month:\n")
print(vol_ratio_monthly.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(12)
meds = [vol_ratio_monthly.loc[m, 'ratio_med'] for m in range(1, 13)]
p25 = [vol_ratio_monthly.loc[m, 'ratio_p25'] for m in range(1, 13)]
p75 = [vol_ratio_monthly.loc[m, 'ratio_p75'] for m in range(1, 13)]

# IQR bands
ax.fill_between(x, p25, p75, alpha=0.2, color=COLORS['hrb'], label='p25–p75')
ax.plot(x, meds, 'o-', color=COLORS['hrb'], lw=2, markersize=8, label='Median', zorder=5)

ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5, label='Ratio = 1.0')

# Shade tax season
for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB RV20 / IWM RV20')
ax.set_title('HRB/IWM Vol Ratio Seasonality by Calendar Month\n(Green shading = tax season)')
ax.legend()

plt.tight_layout()
plt.show()


## 10. Earnings Event Analysis

HRB reports earnings ~8 times per year (4 quarters). The market's reaction should differ by fiscal quarter: a miss on the FQ4 (peak tax) print carries far more information than an FQ1 (dead zone) miss.

We identify earnings dates from large single-day moves (|daily return| > 2 standard deviations) and analyze them by fiscal quarter.


In [ ]:
# Identify likely earnings days: large single-day HRB moves
hrb_ret = df['HRB_ret'].dropna()
ret_std = hrb_ret.std()
ret_mean = hrb_ret.mean()
threshold = 2.0 * ret_std

big_days = df[abs(df['HRB_ret']) > threshold].copy()
big_days['abs_ret'] = abs(big_days['HRB_ret']) * 100
big_days['direction'] = np.where(big_days['HRB_ret'] > 0, 'Up', 'Down')

print(f"Threshold: |return| > {threshold*100:.2f}% (2 sigma)")
print(f"Big move days: {len(big_days)} out of {len(hrb_ret)} trading days ({len(big_days)/len(hrb_ret)*100:.1f}%)")
print(f"\nBy fiscal quarter:")
for fq in ['FQ1', 'FQ2', 'FQ3', 'FQ4']:
    sub = big_days[big_days.fq == fq]
    print(f"  {fq}: {len(sub)} big days, "
          f"median |move| = {sub.abs_ret.median():.1f}%, "
          f"up/down = {(sub.direction == 'Up').sum()}/{(sub.direction == 'Down').sum()}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# -- Panel 1: Distribution of big-day moves by FQ
ax = axes[0]
for fq in ['FQ1', 'FQ2', 'FQ3', 'FQ4']:
    sub = big_days[big_days.fq == fq]
    ax.hist(sub['HRB_ret'] * 100, bins=30, range=(-15, 15), alpha=0.4,
            color=FQ_COLORS[fq], label=f'{fq} (n={len(sub)})', density=True)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('HRB Daily Return (%)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Big Move Days (>2σ) by Fiscal Quarter')
ax.legend()

# -- Panel 2: Median absolute return on big days by FQ
ax = axes[1]
fq_big = big_days.groupby('fq').agg(
    count=('abs_ret', 'count'),
    med_abs_ret=('abs_ret', 'median'),
    mean_abs_ret=('abs_ret', 'mean'),
).reindex(['FQ1', 'FQ2', 'FQ3', 'FQ4'])

ax.bar(range(4), fq_big['med_abs_ret'], 0.6,
       color=[FQ_COLORS[fq] for fq in fq_order], alpha=0.8)
for i, fq in enumerate(fq_order):
    ax.text(i, fq_big.loc[fq, 'med_abs_ret'] + 0.1,
            f'n={int(fq_big.loc[fq, "count"])}', ha='center', fontsize=10)
ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('Median |Return| on Big Days (%)')
ax.set_title('Median Magnitude of Big Moves by Fiscal Quarter')

plt.tight_layout()
plt.show()


## 11. Year-Over-Year Heatmap — Monthly RV20

Each cell shows HRB's average RV20 for that year × month. This reveals whether the seasonal pattern is stable across years or driven by a few outlier years.


In [ ]:
ym_rv = df.dropna(subset=['HRB_rv20']).groupby(['year', 'month'])['HRB_rv20'].median().unstack()

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_rv) * 0.5)))

vals = ym_rv.values
im = ax.imshow(vals, cmap='YlOrRd', aspect='auto',
               vmin=np.nanpercentile(vals, 5), vmax=np.nanpercentile(vals, 95))

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if v > np.nanpercentile(vals, 70) else 'black'
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_rv)))
ax.set_yticklabels(ym_rv.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title('HRB Median RV20 (ann %) by Year × Month')
plt.colorbar(im, shrink=0.8, label='RV20 (ann %)')

plt.tight_layout()
plt.show()


## 12. Year-Over-Year Heatmap — Vol Ratio (HRB / IWM)

Same heatmap but for the vol ratio. This controls for market-wide vol regimes and isolates the HRB-specific seasonal component.


In [ ]:
ym_ratio = df.dropna(subset=['HRB_vs_IWM_rv20']).groupby(['year', 'month'])['HRB_vs_IWM_rv20'].median().unstack()

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_ratio) * 0.5)))

vals = ym_ratio.values
vmax = max(abs(np.nanpercentile(vals, 5) - 1), abs(np.nanpercentile(vals, 95) - 1))
im = ax.imshow(vals, cmap='RdYlGn_r', aspect='auto',
               vmin=1 - vmax, vmax=1 + vmax)

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if abs(v - 1) > vmax * 0.5 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_ratio)))
ax.set_yticklabels(ym_ratio.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title('HRB/IWM Vol Ratio by Year × Month\n(>1 = HRB more volatile than IWM)')
plt.colorbar(im, shrink=0.8, label='HRB RV20 / IWM RV20')

plt.tight_layout()
plt.show()


## 13. Cumulative Return by Fiscal Quarter — Tax Season Alpha?

If we only held HRB during tax season (FQ3+FQ4) vs. only during the dead zone (FQ1+FQ2), what would the return profile look like?


In [ ]:
df_valid = df.dropna(subset=['HRB_ret']).copy()

# Tax season: FQ3 + FQ4 (Nov-Apr)
# Dead zone: FQ1 + FQ2 (May-Oct)
df_valid['tax_season'] = df_valid['fq'].isin(['FQ3', 'FQ4'])

df_valid['hrb_tax_ret'] = np.where(df_valid['tax_season'], df_valid['HRB_ret'], 0)
df_valid['hrb_dead_ret'] = np.where(~df_valid['tax_season'], df_valid['HRB_ret'], 0)
df_valid['hrb_all_ret'] = df_valid['HRB_ret']

# Cumulative returns
df_valid['cum_tax'] = df_valid['hrb_tax_ret'].cumsum()
df_valid['cum_dead'] = df_valid['hrb_dead_ret'].cumsum()
df_valid['cum_all'] = df_valid['hrb_all_ret'].cumsum()
df_valid['cum_iwm'] = df_valid['IWM_ret'].cumsum()

fig, ax = plt.subplots(figsize=(16, 8))
ax.plot(df_valid.index, (np.exp(df_valid['cum_tax']) - 1) * 100,
        color=COLORS['hrb'], lw=1.5, label='HRB: Tax Season Only (Nov–Apr)')
ax.plot(df_valid.index, (np.exp(df_valid['cum_dead']) - 1) * 100,
        color=COLORS['dead_zone'], lw=1.5, label='HRB: Dead Zone Only (May–Oct)')
ax.plot(df_valid.index, (np.exp(df_valid['cum_all']) - 1) * 100,
        color=COLORS['accent'], lw=1, alpha=0.5, label='HRB: All Year')
ax.plot(df_valid.index, (np.exp(df_valid['cum_iwm']) - 1) * 100,
        color=COLORS['iwm'], lw=1, alpha=0.5, label='IWM: All Year')

ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Cumulative Return (%)')
ax.set_title('HRB Cumulative Returns: Tax Season vs Dead Zone')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

# Annualized stats
years = (df_valid.index[-1] - df_valid.index[0]).days / 365.25
for label, col in [('Tax Season', 'cum_tax'), ('Dead Zone', 'cum_dead'), ('All Year', 'cum_all')]:
    total = df_valid[col].iloc[-1]
    ann_ret = (np.exp(total / years) - 1) * 100
    print(f"  HRB {label}: {(np.exp(total)-1)*100:.1f}% total, ~{ann_ret:.1f}% ann")


## 14. Summary Statistics


In [ ]:
print("=" * 70)
print("HRB SEASONALITY SUMMARY")
print("=" * 70)

valid = df.dropna(subset=['HRB_rv20', 'IWM_rv20'])

tax = valid[valid.fq.isin(['FQ3', 'FQ4'])]
dead = valid[valid.fq.isin(['FQ1', 'FQ2'])]

print(f"\nHRB Median RV20:")
print(f"  Tax Season (Nov–Apr):  {tax['HRB_rv20'].median():.1f}%")
print(f"  Dead Zone  (May–Oct):  {dead['HRB_rv20'].median():.1f}%")
print(f"  Ratio (tax/dead):      {tax['HRB_rv20'].median() / dead['HRB_rv20'].median():.2f}x")

print(f"\nIWM Median RV20 (control):")
print(f"  Tax Season (Nov–Apr):  {tax['IWM_rv20'].median():.1f}%")
print(f"  Dead Zone  (May–Oct):  {dead['IWM_rv20'].median():.1f}%")
print(f"  Ratio (tax/dead):      {tax['IWM_rv20'].median() / dead['IWM_rv20'].median():.2f}x")

print(f"\nHRB/IWM Vol Ratio:")
print(f"  Tax Season median:     {tax['HRB_vs_IWM_rv20'].median():.3f}")
print(f"  Dead Zone  median:     {dead['HRB_vs_IWM_rv20'].median():.3f}")

print(f"\nHRB Mean Daily Return (annualized):")
print(f"  Tax Season: {tax['HRB_ret'].mean() * ANN_FACTOR * 100:.1f}%")
print(f"  Dead Zone:  {dead['HRB_ret'].mean() * ANN_FACTOR * 100:.1f}%")
print(f"\n" + "=" * 70)


## 15. Export


In [ ]:
output_path = 'hrb_seasonality_panel.csv'
export_cols = ['HRB_ret', 'IWM_ret', 'SPY_ret',
               'HRB_rv20', 'IWM_rv20', 'SPY_rv20',
               'HRB_rv60', 'IWM_rv60', 'SPY_rv60',
               'HRB_vs_IWM_rv20', 'HRB_vs_SPY_rv20',
               'month', 'year', 'fq']
df[export_cols].to_csv(output_path)
print(f"Exported to {output_path}")
print(f"  {df.shape[0]:,} rows")
